In [0]:

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = "hospital_project"

patients_df = spark.table(
    f"`{catalog}`.{schema}.bronze_patients"
)

appointments_df = spark.table(
    f"`{catalog}`.{schema}.bronze_appointments"
)

treatments_df = spark.table(
    f"`{catalog}`.{schema}.bronze_treatments"
)

display(patients_df)


patient_id,patient_name,age,gender,city,registration_date
P001,Demo Patient 001,32,Male,Chennai,2026-03-04
P002,Demo Patient 002,46,Male,Bengaluru,2026-05-20
P003,Demo Patient 003,29,Other,Mysuru,2026-01-09
P004,Demo Patient 004,21,Male,Hyderabad,2026-03-01
P005,Demo Patient 005,82,Other,Bengaluru,2026-05-24
P006,Demo Patient 006,43,Other,Mysuru,2026-02-26
P007,Demo Patient 007,75,Other,Chennai,2026-01-02
P008,Demo Patient 008,38,Other,Mysuru,2026-03-29
P009,Demo Patient 009,53,Male,Hyderabad,2026-03-28
P010,Demo Patient 010,31,Male,Mysuru,2026-01-25


In [0]:

from pyspark.sql.functions import col, count

duplicates_df = (
    patients_df
    .groupBy("patient_id")
    .agg(count("*").alias("record_count"))
    .filter(col("record_count") > 1)
)

display(duplicates_df)


patient_id,record_count
P005,2
P009,2
P012,2


In [0]:

from pyspark.sql.functions import col, trim

invalid_age_df = patients_df.filter(
    col("age").isNull()
    | (trim(col("age")) == "")
    | (col("age").cast("int") < 0)
    | (col("age").cast("int") > 120)
)

display(invalid_age_df)


patient_id,patient_name,age,gender,city,registration_date
P009,Demo Patient 009,null,Male,Hyderabad,2026-03-28
P012,Demo Patient 012,-5,Other,Bengaluru,2026-04-07


In [0]:

from pyspark.sql.functions import (
    col, trim, to_date, row_number,
    when, lit, concat_ws
)
from pyspark.sql.window import Window


In [0]:

patients_clean_df = (
    patients_df
    .withColumn("patient_id", trim(col("patient_id")))
    .withColumn("patient_name", trim(col("patient_name")))
    .withColumn("gender", trim(col("gender")))
    .withColumn("city", trim(col("city")))
    .withColumn("age", col("age").cast("int"))
    .withColumn(
        "registration_date",
        to_date(col("registration_date"), "yyyy-MM-dd")
    )
)

display(patients_clean_df)


patient_id,patient_name,age,gender,city,registration_date
P001,Demo Patient 001,32,Male,Chennai,2026-03-04
P002,Demo Patient 002,46,Male,Bengaluru,2026-05-20
P003,Demo Patient 003,29,Other,Mysuru,2026-01-09
P004,Demo Patient 004,21,Male,Hyderabad,2026-03-01
P005,Demo Patient 005,82,Other,Bengaluru,2026-05-24
P006,Demo Patient 006,43,Other,Mysuru,2026-02-26
P007,Demo Patient 007,75,Other,Chennai,2026-01-02
P008,Demo Patient 008,38,Other,Mysuru,2026-03-29
P009,Demo Patient 009,53,Male,Hyderabad,2026-03-28
P010,Demo Patient 010,31,Male,Mysuru,2026-01-25


In [0]:

patients_validated_df = (
    patients_clean_df
    .withColumn(
        "rejection_reason",
        concat_ws(
            "; ",
            when(
                col("patient_id").isNull() |
                (col("patient_id") == ""),
                lit("Missing patient ID")
            ),
            when(
                col("age").isNull() |
                (col("age") < 0) |
                (col("age") > 120),
                lit("Invalid age")
            ),
            when(
                col("registration_date").isNull(),
                lit("Invalid registration date")
            )
        )
    )
)

display(
    patients_validated_df.select(
        "patient_id", "age", "rejection_reason"
    )
)


patient_id,age,rejection_reason
P001,32,
P002,46,
P003,29,
P004,21,
P005,82,
P006,43,
P007,75,
P008,38,
P009,53,
P010,31,


In [0]:

valid_patients_df = patients_validated_df.filter(
    col("rejection_reason") == ""
).drop("rejection_reason")

invalid_patients_df = patients_validated_df.filter(
    col("rejection_reason") != ""
)

print("Valid records:", valid_patients_df.count())
print("Rejected records:", invalid_patients_df.count())

display(invalid_patients_df)


Valid records: 51
Rejected records: 2


patient_id,patient_name,age,gender,city,registration_date,rejection_reason
P009,Demo Patient 009,null,Male,Hyderabad,2026-03-28,Invalid age
P012,Demo Patient 012,-5,Other,Bengaluru,2026-04-07,Invalid age


In [0]:

silver_patients_df = valid_patients_df.dropDuplicates()

print(
    "Clean patient records:",
    silver_patients_df.count()
)

display(silver_patients_df)


Clean patient records: 50


patient_id,patient_name,age,gender,city,registration_date
P001,Demo Patient 001,32,Male,Chennai,2026-03-04
P002,Demo Patient 002,46,Male,Bengaluru,2026-05-20
P003,Demo Patient 003,29,Other,Mysuru,2026-01-09
P004,Demo Patient 004,21,Male,Hyderabad,2026-03-01
P005,Demo Patient 005,82,Other,Bengaluru,2026-05-24
P006,Demo Patient 006,43,Other,Mysuru,2026-02-26
P007,Demo Patient 007,75,Other,Chennai,2026-01-02
P008,Demo Patient 008,38,Other,Mysuru,2026-03-29
P009,Demo Patient 009,53,Male,Hyderabad,2026-03-28
P010,Demo Patient 010,31,Male,Mysuru,2026-01-25


In [0]:

silver_patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_patients"
    )

invalid_patients_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_rejected_patients"
    )

print("Silver patient tables created successfully!")


Silver patient tables created successfully!


In [0]:

silver_df = spark.table(
    f"`{catalog}`.{schema}.silver_patients"
)

rejected_df = spark.table(
    f"`{catalog}`.{schema}.silver_rejected_patients"
)

print("Bronze records:", patients_df.count())
print("Silver records:", silver_df.count())
print("Rejected records:", rejected_df.count())
print(
    "Distinct patients:",
    silver_df.select("patient_id").distinct().count()
)


Bronze records: 53
Silver records: 50
Rejected records: 2
Distinct patients: 50


In [0]:

from pyspark.sql.functions import col, trim, to_date

appointments_clean_df = (
    appointments_df
    .withColumn(
        "appointment_id",
        trim(col("appointment_id"))
    )
    .withColumn(
        "patient_id",
        trim(col("patient_id"))
    )
    .withColumn(
        "department",
        trim(col("department"))
    )
    .withColumn(
        "status",
        trim(col("status"))
    )
    .withColumn(
        "appointment_date",
        to_date(col("appointment_date"), "yyyy-MM-dd")
    )
)

display(appointments_clean_df)


appointment_id,patient_id,department,appointment_date,status
A001,P005,Cardiology,2026-07-13,Completed
A002,P033,Orthopedics,2026-07-06,Cancelled
A003,P014,Dermatology,2026-06-17,Cancelled
A004,P016,General Medicine,2026-07-23,Completed
A005,P007,Cardiology,2026-07-26,Scheduled
A006,P028,General Medicine,2026-07-30,Completed
A007,P044,Cardiology,2026-06-08,Cancelled
A008,P047,Neurology,2026-06-14,Completed
A009,P013,Orthopedics,2026-08-08,Cancelled
A010,P009,General Medicine,2026-06-24,Scheduled


In [0]:

appointments_unique_df = (
    appointments_clean_df.dropDuplicates()
)

print("Bronze records:", appointments_df.count())
print("After deduplication:", appointments_unique_df.count())


Bronze records: 92
After deduplication: 91


In [0]:

invalid_appointments_df = (
    appointments_unique_df.alias("a")
    .join(
        silver_patients_df
        .select("patient_id").alias("p"),
        on="patient_id",
        how="left_anti"
    )
)

display(invalid_appointments_df)


patient_id,appointment_id,department,appointment_date,status
P999,A013,Orthopedics,2026-06-22,Cancelled


In [0]:

silver_appointments_df = (
    appointments_unique_df.alias("a")
    .join(
        silver_patients_df
        .select("patient_id").alias("p"),
        on="patient_id",
        how="left_semi"
    )
)

print(
    "Valid appointments:",
    silver_appointments_df.count()
)

display(silver_appointments_df)


Valid appointments: 90


patient_id,appointment_id,department,appointment_date,status
P028,A006,General Medicine,2026-07-30,Completed
P016,A050,Orthopedics,2026-06-23,Cancelled
P023,A055,Neurology,2026-06-30,Completed
P002,A059,Cardiology,2026-07-04,Completed
P038,A060,Neurology,2026-06-05,Completed
P036,A071,Cardiology,2026-07-09,Scheduled
P019,A075,Dermatology,2026-08-19,Scheduled
P013,A009,Orthopedics,2026-08-08,Cancelled
P028,A017,Dermatology,2026-08-02,Completed
P033,A021,Dermatology,2026-06-21,Completed


In [0]:

silver_appointments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_appointments"
    )

invalid_appointments_df.withColumn(
    "rejection_reason",
    lit("Patient ID not found")
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_rejected_appointments"
    )

print("Silver appointment tables created!")


Silver appointment tables created!


In [0]:

print("Bronze:", appointments_df.count())
print("Silver:", silver_appointments_df.count())
print("Rejected:", invalid_appointments_df.count())

print(
    "Unique appointment IDs:",
    silver_appointments_df
    .select("appointment_id")
    .distinct()
    .count()
)


Bronze: 92
Silver: 90
Rejected: 1
Unique appointment IDs: 90


In [0]:

from pyspark.sql.functions import (
    col, trim, when, lit
)

catalog = spark.sql("SELECT current_catalog()").first()[0]
schema = "hospital_project"

treatments_df = spark.table(
    f"`{catalog}`.{schema}.bronze_treatments"
)

appointments_df = spark.table(
    f"`{catalog}`.{schema}.silver_appointments"
)

print("Bronze treatments:", treatments_df.count())
print("Silver appointments:", appointments_df.count())


Bronze treatments: 44
Silver appointments: 90


In [0]:

from pyspark.sql.functions import expr

treatments_clean_df = (
    treatments_df
    .withColumn("treatment_id", trim(col("treatment_id")))
    .withColumn("appointment_id", trim(col("appointment_id")))
    .withColumn("treatment_type", trim(col("treatment_type")))
    .withColumn(
        "cost",
        expr("try_cast(cost AS DECIMAL(10,2))")
    )
)

display(treatments_clean_df)


treatment_id,appointment_id,treatment_type,cost
T001,A001,Blood Test,1200.00
T002,A004,ECG,1200.00
T003,A006,Physiotherapy,500.00
T004,A008,Blood Test,750.00
T005,A012,Blood Test,1800.00
T006,A015,Blood Test,3200.00
T007,A017,Blood Test,500.00
T008,A018,X-Ray,1800.00
T009,A021,ECG,2500.00
T010,A022,X-Ray,1800.00


In [0]:

invalid_cost_df = treatments_clean_df.filter(
    col("cost").isNull() |
    (col("cost") < 0)
)

print("Invalid treatment costs:", invalid_cost_df.count())

display(invalid_cost_df)


Invalid treatment costs: 1


treatment_id,appointment_id,treatment_type,cost
T999,A008,Blood Test,-100.00


In [0]:

valid_appointment_ids = (
    appointments_df
    .select("appointment_id")
    .distinct()
    .withColumn("appointment_exists", lit(True))
)

treatments_validated_df = (
    treatments_clean_df
    .join(
        valid_appointment_ids,
        on="appointment_id",
        how="left"
    )
    .withColumn(
        "rejection_reason",
        when(
            col("treatment_id").isNull() |
            (col("treatment_id") == ""),
            "Missing treatment ID"
        )
        .when(
            col("cost").isNull() |
            (col("cost") < 0),
            "Invalid treatment cost"
        )
        .when(
            col("appointment_exists").isNull(),
            "Appointment ID not found"
        )
        .otherwise(None)
    )
    .drop("appointment_exists")
)

display(treatments_validated_df)


appointment_id,treatment_id,treatment_type,cost,rejection_reason
A001,T001,Blood Test,1200.00,null
A004,T002,ECG,1200.00,null
A006,T003,Physiotherapy,500.00,null
A008,T004,Blood Test,750.00,null
A012,T005,Blood Test,1800.00,null
A015,T006,Blood Test,3200.00,null
A017,T007,Blood Test,500.00,null
A018,T008,X-Ray,1800.00,null
A021,T009,ECG,2500.00,null
A022,T010,X-Ray,1800.00,null


In [0]:

silver_treatments_df = (
    treatments_validated_df
    .filter(col("rejection_reason").isNull())
    .drop("rejection_reason")
    .dropDuplicates()
)

rejected_treatments_df = (
    treatments_validated_df
    .filter(col("rejection_reason").isNotNull())
)

print("Valid treatments:", silver_treatments_df.count())
print("Rejected treatments:", rejected_treatments_df.count())


Valid treatments: 43
Rejected treatments: 1


In [0]:

silver_treatments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_treatments"
    )

rejected_treatments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        f"`{catalog}`.{schema}.silver_rejected_treatments"
    )

print("Silver treatment tables created successfully!")


Silver treatment tables created successfully!


In [0]:

tables = [
    "silver_patients",
    "silver_appointments",
    "silver_treatments"
]

for table in tables:
    df = spark.table(f"`{catalog}`.{schema}.{table}")
    print(f"{table}: {df.count()} records")


silver_patients: 50 records
silver_appointments: 90 records
silver_treatments: 43 records
